In [10]:
import pandas as pd  # Load pandas.

raw = pd.read_csv("C:\\Users\\twizz\\Downloads\\teaching_prices_messy.csv")  # Read the original dataset.
raw_before = raw.copy(deep=True)  # Preserve a reference for the final check.
clean = raw.copy(deep=True)  # Make the working copy.
original_columns = raw.columns.tolist()  # Remember the six supplied columns.

print("Starting rows:", len(raw))  # Display the original record count.

Starting rows: 97


In [11]:
clean["missing_price"] = clean["price"].isna()  # Flag missing prices as True.

print("Missing prices:", clean["missing_price"].sum())  # Count flagged rows.
print(clean.loc[clean["missing_price"]])  # Display those rows.

Missing prices: 1
         date     item    city price                       unit  \
5  2026-01-01  Pulse C  City 2   NaN  PKR per kg (illustrative)   

                                          data_label  missing_price  
5  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...           True  


In [12]:
duplicate_rows = clean.duplicated(
    subset=original_columns,
    keep="first"
)  # Compare all six original fields and mark additional identical copies.

excluded_duplicates = clean.loc[duplicate_rows].copy()  # Preserve excluded records.
print(excluded_duplicates)  # Show exactly what will be removed.

clean = clean.loc[~duplicate_rows].copy()  # Keep rows that are not extra copies.
print("Rows after removing exact duplicates:", len(clean))  # Check the count.

          date      item    city price                       unit  \
96  2026-01-08  Staple A  City 1  99.5  PKR per kg (illustrative)   

                                           data_label  missing_price  
96  ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...          False  
Rows after removing exact duplicates: 96


In [13]:
item_check = clean[["date", "item", "city", "price"]].copy()  # Make a test table.
item_check["item_original"] = item_check["item"]  # Preserve original labels.

item_check["item"] = item_check["item"].replace(
    {"oil b": "Oil B"}
)  # Test the proposed capitalization change in this separate table.

conflicts = item_check.duplicated(
    subset=["date", "item", "city"],
    keep=False
)  # Flag every record sharing a key after the proposed change.

print(item_check.loc[conflicts])  # Inspect records whose keys would conflict.

clean["item_label_review"] = clean["item"].eq("oil b")  # Flag the uncertain label.
clean["key_conflict_if_normalized"] = conflicts  # Record the conflict warning.

          date   item    city  price item_original
40  2026-02-05  Oil B  City 1  108.5         oil b
42  2026-02-05  Oil B  City 1  318.0         Oil B


In [14]:
price_text = clean["price"].astype("string").str.strip()  # Prepare temporary text.
direct_numeric = pd.to_numeric(price_text, errors="coerce")  # Test conversion.

conversion_problem = price_text.notna() & direct_numeric.isna()  # Find failed nonmissing values.
print(clean.loc[conversion_problem, ["date", "item", "city", "price"]])  # Show the originals.

          date      item    city  price
17  2026-01-15  Staple A  City 2  1,250


In [16]:
clean["price_original"] = clean["price"]  # Preserve the source values.

price_text = price_text.str.replace(",", "", regex=False)  # Remove verified thousands separators.
clean["price"] = pd.to_numeric(price_text, errors="raise")  # Convert, stopping on unrecognized values.

print(clean["price"].dtype)  # Check the resulting numeric type.
print("Missing prices after conversion:", clean["price"].isna().sum())  # Check for unexpected missing values.

Float64
Missing prices after conversion: 1


In [17]:
date_check = pd.to_datetime(
    clean["date"],
    format="%Y-%m-%d",
    errors="coerce"
)  # Test the expected date format using a temporary series.

clean["invalid_date"] = clean["date"].notna() & date_check.isna()  # Flag failed nonmissing dates.
print(clean.loc[clean["invalid_date"], ["date", "item", "city"]])  # Display failures.

if clean["invalid_date"].any():  # Check whether any dates failed.
    print("Investigate flagged dates; the date column has not been changed.")
else:
    clean["date_original"] = clean["date"]  # Preserve the original date strings.
    clean["date"] = date_check  # Convert only when validation succeeds.
    print("Date conversion completed.")

Empty DataFrame
Columns: [date, item, city]
Index: []
Date conversion completed.


In [19]:
row_counts = pd.DataFrame(
    {"before": [len(raw)], "after": [len(clean)]},
    index=["rows"]
)  # Compare record counts.
print(row_counts)

missing_counts = pd.DataFrame({
    "before": raw[original_columns].isna().sum(),
    "after": clean[original_columns].isna().sum()
})  # Compare missing values in the original six columns.
print(missing_counts)

data_types = pd.DataFrame({
    "before": raw[original_columns].dtypes.astype(str),
    "after": clean[original_columns].dtypes.astype(str)
})  # Compare column types.
print(data_types)

item_counts = pd.DataFrame({
    "before": raw["item"].value_counts(dropna=False),
    "after": clean["item"].value_counts(dropna=False)
}).fillna(0).astype(int)  # Compare record counts for each item label.
print(item_counts)

print(clean["price"].describe())  # Inspect the numeric price summary after conversion.

pd.testing.assert_frame_equal(raw, raw_before)  # Verify that raw is unchanged.
assert len(raw) == len(clean) + len(excluded_duplicates)  # Account for every original row.
print("Original data unchanged; row counts reconcile.")

      before  after
rows      97     96
            before  after
date             0      0
item             0      0
city             0      0
price            1      1
unit             0      0
data_label       0      0
           before           after
date          str  datetime64[us]
item          str             str
city          str             str
price         str         Float64
unit          str             str
data_label    str             str
             before  after
item                      
Oil B            24     24
Pulse C          24     24
Staple A         24     23
Sweetener D      24     24
oil b             1      1
count           95.0
mean      317.731579
std      1012.659212
min             99.5
25%            151.0
50%            200.0
75%            304.0
max           9999.0
Name: price, dtype: Float64
Original data unchanged; row counts reconcile.
